In [1]:
import pandas as pd

# Load
tx = pd.read_csv("transactions.csv") 
acc = pd.read_csv("accounts.csv")

# 1. Basic shape & label distribution
print(tx.shape)
print(tx['Is Laundering'].value_counts(normalize=True))

(5078345, 11)
Is Laundering
0    0.998981
1    0.001019
Name: proportion, dtype: float64


In [2]:
# 2. Time range
tx['Timestamp'] = pd.to_datetime(tx['Timestamp'])
print(tx['Timestamp'].min(), tx['Timestamp'].max())

2022-09-01 00:00:00 2022-09-18 16:18:00


In [3]:
# 3. Amount distribution
print(tx['Amount Paid'].describe())
print(tx['Amount Paid'].quantile([0.5, 0.9, 0.95, 0.99]))

count    5.078345e+06
mean     4.509273e+06
std      8.697728e+08
min      1.000000e-06
25%      1.844800e+02
50%      1.414540e+03
75%      1.229784e+04
max      1.046302e+12
Name: Amount Paid, dtype: float64
0.50    1.414540e+03
0.90    1.372753e+05
0.95    6.237572e+05
0.99    1.352453e+07
Name: Amount Paid, dtype: float64


In [4]:
# 4. Payment format distribution
print(tx['Payment Format'].value_counts())

Payment Format
Cheque          1864331
Credit Card     1323324
ACH              600797
Cash             490891
Reinvestment     481056
Wire             171855
Bitcoin          146091
Name: count, dtype: int64


In [5]:
# 5. Currency distribution & mismatch rate
print(tx['Receiving Currency'].value_counts())
print(tx['Payment Currency'].value_counts())
print((tx['Receiving Currency'] != tx['Payment Currency']).mean())

Receiving Currency
US Dollar            1879341
Euro                 1172017
Swiss Franc           237884
Yuan                  206551
Shekel                194988
Rupee                 192065
UK Pound              181255
Ruble                 157361
Yen                   156319
Bitcoin               148151
Canadian Dollar       141357
Australian Dollar     138511
Mexican Peso          111030
Saudi Riyal            89971
Brazil Real            71544
Name: count, dtype: int64
Payment Currency
US Dollar            1895172
Euro                 1168297
Swiss Franc           234860
Yuan                  213752
Shekel                192184
Rupee                 190202
UK Pound              180738
Yen                   155209
Ruble                 155178
Bitcoin               146066
Canadian Dollar       140042
Australian Dollar     136769
Mexican Peso          110159
Saudi Riyal            89014
Brazil Real            70703
Name: count, dtype: int64
0.014211322783308342


In [6]:
# 6. Laundering rate BY payment format and BY currency-mismatch flag
tx['currency_mismatch'] = tx['Receiving Currency'] != tx['Payment Currency']
print(tx.groupby('Payment Format')['Is Laundering'].mean())
print(tx.groupby('currency_mismatch')['Is Laundering'].mean())

Payment Format
ACH             0.007462
Bitcoin         0.000383
Cash            0.000220
Cheque          0.000174
Credit Card     0.000156
Reinvestment    0.000000
Wire            0.000000
Name: Is Laundering, dtype: float64
currency_mismatch
False    0.001034
True     0.000000
Name: Is Laundering, dtype: float64


In [7]:
# 7. Account-level transaction frequency (for velocity rule baseline)
acct_freq = tx.groupby('Account').size().describe()
print(acct_freq)

count    496995.000000
mean         10.218101
std         290.284113
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max      168672.000000
dtype: float64


In [8]:
# 8. Laundering rate by amount bucket (where does laundering concentrate in the distribution?)
tx['amount_bucket'] = pd.qcut(tx['Amount Paid'], q=10, duplicates='drop')
print(tx.groupby('amount_bucket')['Is Laundering'].mean())

amount_bucket
(-0.000999, 25.42]               0.000167
(25.42, 118.04]                  0.000167
(118.04, 279.84]                 0.000278
(279.84, 638.68]                 0.000376
(638.68, 1414.54]                0.000652
(1414.54, 3029.81]               0.001178
(3029.81, 7311.16]               0.001934
(7311.16, 21539.97]              0.003153
(21539.97, 137275.31]            0.001227
(137275.31, 1046302363293.48]    0.001063
Name: Is Laundering, dtype: float64


/var/folders/gk/3862nl_50qnbbb0lrm7wlstm0000gn/T/ipykernel_7000/3416167807.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tx.groupby('amount_bucket')['Is Laundering'].mean())


In [9]:
# 9. Same-bank vs cross-bank laundering rate
tx['same_bank'] = tx['From Bank'] == tx['To Bank']
print(tx.groupby('same_bank')['Is Laundering'].mean())

same_bank
False    0.001157
True     0.000149
Name: Is Laundering, dtype: float64


In [10]:
# 10. Hour-of-day pattern
tx['hour'] = tx['Timestamp'].dt.hour
print(tx.groupby('hour')['Is Laundering'].mean())

hour
0     0.000277
1     0.000785
2     0.000854
3     0.000756
4     0.000797
5     0.000970
6     0.001065
7     0.001009
8     0.001337
9     0.001125
10    0.001211
11    0.001525
12    0.001741
13    0.001517
14    0.001447
15    0.001350
16    0.001608
17    0.001331
18    0.001319
19    0.001198
20    0.000695
21    0.000801
22    0.000667
23    0.000776
Name: Is Laundering, dtype: float64


In [11]:
# 11. Hub account exclusion check — does laundering happen on high-volume hub accounts or low-volume ones?
acct_tx_count = tx.groupby('Account').size().rename('acct_freq')
tx_with_freq = tx.merge(acct_tx_count, left_on='Account', right_index=True)
tx_with_freq['freq_bucket'] = pd.qcut(tx_with_freq['acct_freq'], q=5, duplicates='drop')
print(tx_with_freq.groupby('freq_bucket')['Is Laundering'].mean())

freq_bucket
(0.999, 15.0]       0.002421
(15.0, 31.0]        0.001071
(31.0, 53.0]        0.000432
(53.0, 95.0]        0.000281
(95.0, 168672.0]    0.000857
Name: Is Laundering, dtype: float64


/var/folders/gk/3862nl_50qnbbb0lrm7wlstm0000gn/T/ipykernel_7000/2145611759.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tx_with_freq.groupby('freq_bucket')['Is Laundering'].mean())


In [12]:
# 12. Repeat account-pair behavior (does laundering happen between accounts that transact once, or repeatedly?)
pair_freq = tx.groupby(['Account', 'Account.1']).size().rename('pair_freq') 
tx_pair = tx.merge(pair_freq, left_on=['Account','Account.1'], right_index=True)
print(tx_pair.groupby(pd.qcut(tx_pair['pair_freq'], q=5, duplicates='drop'))['Is Laundering'].mean())

pair_freq
(0.999, 3.0]    0.003989
(3.0, 14.0]     0.000271
(14.0, 20.0]    0.000203
(20.0, 21.0]    0.000189
(21.0, 89.0]    0.000158
Name: Is Laundering, dtype: float64


/var/folders/gk/3862nl_50qnbbb0lrm7wlstm0000gn/T/ipykernel_7000/1582105785.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tx_pair.groupby(pd.qcut(tx_pair['pair_freq'], q=5, duplicates='drop'))['Is Laundering'].mean())


In [14]:
# 13. Build all key flags on the full transaction table
tx['cross_bank'] = tx['From Bank'] != tx['To Bank']
tx['is_ach'] = tx['Payment Format'] == 'ACH'
tx['mid_amount'] = tx['Amount Paid'].between(3029.81, 21539.97)

pair_freq = tx.groupby(['Account', 'Account.1']).size().rename('pair_freq')
tx = tx.merge(pair_freq, left_on=['Account', 'Account.1'], right_index=True)
tx['rare_pair'] = tx['pair_freq'] <= 3

acct_freq = tx.groupby('Account').size().rename('acct_freq_2')
tx = tx.merge(acct_freq, left_on='Account', right_index=True)
tx['low_activity_acct'] = tx['acct_freq_2'] <= 15

# 14. Stack the flags and check laundering rate by number of flags triggered
tx['flag_count'] = tx[['cross_bank','is_ach','mid_amount','rare_pair','low_activity_acct']].sum(axis=1)
print(tx.groupby('flag_count')['Is Laundering'].agg(['mean','count']))

                mean    count
flag_count                   
0           0.000000    87134
1           0.000180  2738050
2           0.000193  1608603
3           0.001952   450920
4           0.014904   151036
5           0.029154    42602


In [15]:
# 15. Specifically test the two strongest signals together: rare_pair + cross_bank
print(tx.groupby(['rare_pair','cross_bank'])['Is Laundering'].agg(['mean','count']))

                          mean    count
rare_pair cross_bank                   
False     False       0.000117   188800
          True        0.000223  3810545
True      False       0.000161   502532
          True        0.007326   576468


In [16]:
# 16. Precision/recall preview at flag_count >= 2 and >= 3
for threshold in [1, 2, 3, 4]:
    flagged = tx['flag_count'] >= threshold
    precision = tx.loc[flagged, 'Is Laundering'].mean()
    recall = tx.loc[flagged, 'Is Laundering'].sum() / tx['Is Laundering'].sum()
    volume = flagged.mean()
    print(f"threshold>={threshold}: precision={precision:.4f}, recall={recall:.4f}, volume_flagged={volume:.4f}")

threshold>=1: precision=0.0010, recall=1.0000, volume_flagged=0.9828
threshold>=2: precision=0.0021, recall=0.9046, volume_flagged=0.4437
threshold>=3: precision=0.0068, recall=0.8447, volume_flagged=0.1269
threshold>=4: precision=0.0180, recall=0.6747, volume_flagged=0.0381


In [18]:
pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 7.8 MB/s  0:00:02 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [19]:
tx['risk_score'] = (5*tx['rare_pair'].astype(int) + 3*tx['cross_bank'].astype(int) +
                    3*tx['is_ach'].astype(int) + 1*tx['mid_amount'].astype(int) +
                    1*tx['low_activity_acct'].astype(int))

print(tx.groupby('risk_score')['Is Laundering'].agg(['mean','count']))

for threshold in [3,4,5,6,7,8,9,10]:
    flagged = tx['risk_score'] >= threshold
    precision = tx.loc[flagged, 'Is Laundering'].mean()
    recall = tx.loc[flagged, 'Is Laundering'].sum() / tx['Is Laundering'].sum()
    volume = flagged.mean()
    print(f"threshold>={threshold}: precision={precision:.4f}, recall={recall:.4f}, volume_flagged={volume:.4f}")

                mean    count
risk_score                   
0           0.000000    87134
1           0.000000    24628
2           0.000000      946
3           0.000188  2624025
4           0.000154   863347
5           0.000014   143374
6           0.000194   614012
7           0.000831   131174
8           0.000557   134715
9           0.000354   251387
10          0.000576    33006
11          0.047281    14382
12          0.019496   113613
13          0.029154    42602
threshold>=3: precision=0.0010, recall=1.0000, volume_flagged=0.9778
threshold>=4: precision=0.0020, recall=0.9046, volume_flagged=0.4611
threshold>=5: precision=0.0031, recall=0.8789, volume_flagged=0.2911
threshold>=6: precision=0.0034, recall=0.8785, volume_flagged=0.2629
threshold>=7: precision=0.0061, recall=0.8555, volume_flagged=0.1420
threshold>=8: precision=0.0073, recall=0.8345, volume_flagged=0.1161
threshold>=9: precision=0.0093, recall=0.8200, volume_flagged=0.0896
threshold>=10: precision=0.0204, reca

In [2]:
import duckdb

In [4]:
con = duckdb.connect()

In [6]:
con.execute("""
Create table tx as 
Select * 
from read_csv_auto('transactions.csv');
""")

In [7]:
result = con.execute("Describe tx").fetchdf()
print(result)

           column_name column_type null   key default extra
0            Timestamp   TIMESTAMP  YES  None    None  None
1            From Bank     VARCHAR  YES  None    None  None
2              Account     VARCHAR  YES  None    None  None
3              To Bank     VARCHAR  YES  None    None  None
4            Account_1     VARCHAR  YES  None    None  None
5      Amount Received      DOUBLE  YES  None    None  None
6   Receiving Currency     VARCHAR  YES  None    None  None
7          Amount Paid      DOUBLE  YES  None    None  None
8     Payment Currency     VARCHAR  YES  None    None  None
9       Payment Format     VARCHAR  YES  None    None  None
10       Is Laundering      BIGINT  YES  None    None  None


In [9]:
con.execute("""
Alter Table tx ADD COLUMN cross_bank Boolean;
Update tx SET cross_bank = ("From Bank" <> "To Bank");
""")

In [10]:
con.execute("""
ALTER TABLE tx ADD COLUMN is_ach BOOLEAN;
UPDATE tx SET is_ach = ("Payment Format" = 'ACH');
""")

In [11]:
con.execute("""
ALTER TABLE tx ADD COLUMN mid_amount BOOLEAN;
UPDATE tx SET mid_amount = ("Amount Paid" Between 3029.81 and 21539.97);
""")

In [14]:
con.execute("""
Alter Table tx ADD COLUMN pair_freq INTEGER;
Update tx
SET pair_freq = sub.cnt
FROM (
SELECT "Account", "Account_1", Count(*) as cnt 
FROM tx
Group By "Account", "Account_1"
            ) sub
where tx."Account" = sub."Account" AND tx."Account_1" = sub."Account_1";

Alter TABLE tx ADD column rare_pair BOOLEAN;
Update tx SET rare_pair = (pair_freq <=3);
""")

In [16]:
con.execute("""
Alter table tx ADD column acct_freq INteger;
Update tx
Set acct_freq = sub.cnt
FROM (
Select "Account", COunt(*) as cnt
From tx
Group by "Account"
            ) sub
where tx."Account" = sub."Account";

ALTER table tx ADD COLUmn low_activity_account Boolean;
UPdate tx SET low_activity_account = (acct_freq <=15);
""")

In [22]:
con.execute("""
ALTER TABLE tx DROP COLUMN IF EXISTS risk_score;

ALTER TABLE tx ADD COLUMN risk_score INTEGER;

UPDATE tx
SET risk_score = 
    (5 * CAST(rare_pair AS INTEGER)) +
    (3 * CAST(cross_bank AS INTEGER)) +
    (3 * CAST(is_ach AS INTEGER)) +
    (1 * CAST(mid_amount AS INTEGER)) +
    (1 * CAST(low_activity_account AS INTEGER));
""")

In [23]:
con.execute("""
ALTER TABLE tx ADD COlumn alert_tier VARCHAR;
UPDATE tx 
SET alert_tier = 
CASE 
WHEN rare_pair and cross_bank THEN 'Tier 2 - Priority (rule B override)'
WHen risk_score >= 11 THEN 'Tier 2 - Priority'
WHen risk_score >= 9 THEN 'Tier 1 - Investigate'
ELse 'No alert'
End;
""")

In [25]:
con.execute("""
Select alert_tier,
COunt(*) as total_transactions,
SUM(CASt("Is Laundering" AS INteger)) as true_launderers_caught,
AVG(CAST("Is Laundering" As Integer)) as precision
FROM tx 
GROUP by alert_tier
Order by alert_tier;
""").fetchdf()

,alert_tier,total_transactions,true_launderers_caught,precision
0,No alert,4495000,899.0,0.000200
1,Tier 1 - Investigate,6876,55.0,0.007999
2,Tier 2 - Priority (rule B override),576469,4223.0,0.007326


In [27]:
con.execute("""
Select 
    Case
        WHEN rare_pair and cross_bank then 'Caught by Rule B'
        WHEN risk_score >= 9 Then 'Caught by Rule A only'
        Else 'No Alert'
    End as catch_source,
    COunt(*) as total_transactions,
    SUM(Cast("Is Laundering" As Integer)) as true_launderers_caught,
    AVG(Cast("Is Laundering" As Integer)) as precision
FROM tx
Group by catch_source
Order by catch_source;
""").fetchdf()

,catch_source,total_transactions,true_launderers_caught,precision
0,Caught by Rule A only,6876,55.0,0.007999
1,Caught by Rule B,576469,4223.0,0.007326
2,No Alert,4495000,899.0,0.000200


In [28]:
con.execute("""
SELECT 
    rare_pair, cross_bank, is_ach, mid_amount, low_activity_account,
    risk_score,
    COUNT(*) AS total_transactions,
    SUM(CAST("Is Laundering" AS INTEGER)) AS true_launderers_caught
FROM tx
WHERE risk_score >= 9 
  AND NOT (rare_pair AND cross_bank)
GROUP BY rare_pair, cross_bank, is_ach, mid_amount, low_activity_account, risk_score
ORDER BY true_launderers_caught DESC;
""").fetchdf()

,rare_pair,cross_bank,is_ach,mid_amount,low_activity_account,risk_score,total_transactions,true_launderers_caught
0,True,False,True,True,False,9,802,35.0
1,True,False,True,True,True,10,1919,10.0
2,True,False,True,False,True,9,4155,10.0
